**Feature Selection Method:** WOA   
**Dataset:** QUT-DV25 (Pattern)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression

In [ ]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "Pattern"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()


In [ ]:
data.head()

,Package_Name,Pattern_1,Pattern_2,Pattern_3,Pattern_4,Pattern_5,Pattern_6,Pattern_7,Pattern_8,Pattern_9,Pattern_10,Level
0,10Cent10-999.0.4.tar.gz,newfstatat -> newfstatat -> newfstatat,fstat -> ioctl -> lseek,openat -> fstat -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> openat -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,fstat -> ioctl -> lseek -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,ioctl -> lseek -> lseek -> error=ENOTTY -> no-fd,read -> read -> close -> no-error -> no-fd,1
1,10Cent11-999.0.4.tar.gz,newfstatat -> newfstatat -> newfstatat,read -> read -> read,newfstatat -> openat -> fstat,ioctl -> ioctl -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,ioctl -> ioctl -> ioctl -> no-error -> no-fd,fcntl -> fstat -> fcntl -> no-error -> no-fd,1
2,11Cent-999.0.0.tar.gz,newfstatat -> newfstatat -> newfstatat,newfstatat -> newfstatat -> openat,read -> read -> read,newfstatat -> openat -> fstat,fstat -> ioctl -> lseek,newfstatat -> newfstatat -> newfstatat -> no-e...,newfstatat -> newfstatat -> openat -> no-error...,read -> read -> read -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,1
3,11Cent-999.0.1.tar.gz,newfstatat -> newfstatat -> newfstatat,read -> read -> read,newfstatat -> openat -> fstat,openat -> fstat -> fcntl,fstat -> fcntl -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,openat -> fstat -> fcntl -> no-error -> no-fd,fstat -> fcntl -> fstat -> no-error -> no-fd,1
4,11Cent-999.0.2.tar.gz,newfstatat -> newfstatat -> newfstatat,read -> read -> read,newfstatat -> openat -> fstat,read -> poll -> read,poll -> read -> read,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,read -> poll -> read -> error=EAGAIN -> no-fd,newfstatat -> newfstatat -> openat -> no-error...,1


In [ ]:

X_df = data.drop(columns=['Level']).copy()
y = data['Level']

# Encode label if needed
if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Identify categorical columns (strings)
cat_cols = X_df.select_dtypes(include='object').columns
num_cols = X_df.select_dtypes(include=['int64','float64']).columns

# Simple Label Encoding for categorical columns
for col in cat_cols:
    X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

# Now all columns are numeric
X = X_df.values
feature_names = list(X_df.columns)

# =====================
# Fitness Function
# =====================
def fitness_function(selected_features, X_train, y_train, X_val, y_val):
    if np.sum(selected_features) == 0:
        return 0
    selected_cols = np.where(selected_features == 1)[0]
    model = LogisticRegression(max_iter=500)
    model.fit(X_train[:, selected_cols], y_train)
    preds = model.predict(X_val[:, selected_cols])
    return accuracy_score(y_val, preds)

# =====================
# WOA Feature Selection (20 iterations)
# =====================
def woa_feature_selection(X, y, num_whales=5, max_iter=20):
    n_features = X.shape[1]

    # Initialize whale population (binary feature subsets)
    whales = np.random.randint(2, size=(num_whales, n_features))

    # Train/validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Evaluate fitness of each whale
    fitness = np.array([fitness_function(w, X_train, y_train, X_val, y_val) for w in whales])
    best_idx = np.argmax(fitness)
    best_whale = whales[best_idx].copy()
    best_score = fitness[best_idx]

    # Main WOA loop
    for t in range(max_iter):
        a = 2 - t * (2 / max_iter)  # linearly decreases from 2 to 0
        for i in range(num_whales):
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2 * a * r1 - a
            C = 2 * r2
            p = np.random.rand()

            if p < 0.5:
                if abs(A) < 1:
                    # Encircling prey (move towards best whale)
                    D = abs(C * best_whale - whales[i])
                    new_position = best_whale - A * D
                else:
                    # Search for prey (random whale)
                    rand_whale = whales[np.random.randint(num_whales)]
                    D = abs(C * rand_whale - whales[i])
                    new_position = rand_whale - A * D
            else:
                # Spiral update (bubble-net attacking)
                D = abs(best_whale - whales[i])
                l = np.random.uniform(-1, 1)
                new_position = D * np.exp(0.5 * l) * np.cos(2 * np.pi * l) + best_whale

            # Sigmoid transfer function for binary update
            sigmoid = 1 / (1 + np.exp(-new_position))
            whales[i] = (np.random.rand(n_features) < sigmoid).astype(int)

            # Evaluate new position
            score = fitness_function(whales[i], X_train, y_train, X_val, y_val)
            if score > fitness[i]:
                fitness[i] = score
                if score > best_score:
                    best_score = score
                    best_whale = whales[i].copy()

        selected = [feature_names[j] for j in range(n_features) if best_whale[j] == 1]
        dropped = [feature_names[j] for j in range(n_features) if best_whale[j] == 0]
        print(f"Iteration {t+1}/{max_iter}, Best Accuracy: {best_score:.4f}")
        print(f"Selected features: {selected}")
        print(f"Dropped features: {dropped}\n")

    # Final features after all iterations
    final_selected = [feature_names[j] for j in range(n_features) if best_whale[j] == 1]
    final_dropped = [feature_names[j] for j in range(n_features) if best_whale[j] == 0]
    print("===== FINAL RESULTS AFTER 20 ITERATIONS =====")
    print(f"Best Accuracy: {best_score:.4f}")
    print(f"Final Selected features: {final_selected}")
    print(f"Final Dropped features: {final_dropped}")

    return best_whale, best_score

best_features, best_score = woa_feature_selection(X, y, num_whales=5, max_iter=20)


Iteration 1/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 2/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 3/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 4/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 5/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 6/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 7/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_7', 'Pattern_8', 'Pattern_10']
Dropped features: ['Pattern_1', 'Pattern_6', 'Pattern_9']

Iteration 8/20, Best Accuracy: 0.6504
Selected features: ['Package_Name', 'Pattern_2', 'Pattern_3', 

In [ ]:
selected_features = ['Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_8', 'Pattern_10']
